In [4]:
# Ключ от Gemini
GOOGLE_API_KEY = input("Введите ваш Gemini API Key: ")

In [5]:
!pip install pymorphy3 nltk
!pip install rank_bm25
!pip install sentence-transformers
!pip install google-genai

In [6]:
import json, re
from pathlib import Path
import nltk
import pymorphy3
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
import torch
from sentence_transformers import SentenceTransformer, util
from google import genai
from google.genai import types
import shutil

In [7]:
# Удаляем старую директорию во избежании ошибок
chunks_dir = Path("./chunks")
if chunks_dir.exists():
    shutil.rmtree(chunks_dir)
    print("Старая папка 'chunks' удалена.")

Делаем из нашей базы знананий набор чанков. Нам нужны как raw чанки для подачи в LLM, так и очищенные от ссылок и спец символов, для работы ретриверов.

In [8]:
def clean_for_search(text: str) -> str:
    # Превращаем пайпы таблиц в точки с запятой
    text = text.replace('|', ';')
    
    # Вырезаем любые остатки картинок, битых ссылок и путей к файлам (.png, .jpg)
    text = re.sub(r'!\[[^\]]*\](?:\([^\)]*\))?', '', text) # Полные и битые теги ![alt](url)
    text = re.sub(r'\[(.*?)\]\([^\)]*\)', r'\1', text)    # Ссылки [текст](url) оставляем только "текст"
    text = re.sub(r'(?i)\b(?:https?|tps)://\S+', '', text) # Любые URL/битые урлы
    text = re.sub(r'(?i)\b\S*?\.(?:png|jpg|jpeg|gif|webp)\b\S*', '', text) # Файлы картинок в тексте
    text = re.sub(r'(?i)"[^"]+\.(?:png|jpg|jpeg|gif|webp)"', '', text)     # Картинки в кавычках
    
    # Стираем маркеры Markdown
    text = re.sub(r'[#*_~`•\-]', ' ', text)
    text = text.replace('\xa0', ' ')
    
    # Схлопываем множественные разделители таблиц подряд в один
    text = re.sub(r';[\s;]*;', ';', text)
    
    # Превращаем всё в одну чистую строку без лишних пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    return text.strip('; ') # Убираем ';' на краях, если остались

In [9]:
def chunk_by_lines(text: str, max_lines=30, overlap_lines=4) -> list:
    lines = text.splitlines()
    total_lines = len(lines)
    chunks = []
    
    start = 0
    while start < total_lines:
        end = min(start + max_lines, total_lines)
        
        # Собираем строки текущего чанка
        chunk_text = "\n".join(lines[start:end]).strip()
        if chunk_text:
            chunks.append(chunk_text)
            
        if end == total_lines:
            break
            
        # Определяем дефолтную точку следующего старта с учетом оверлапа
        target_start = end - overlap_lines
        
        # Сканируем зону перекрытия в поисках начала логического предложения
        found_good_start = False
        for i in range(target_start, end):
            line = lines[i].strip()
            # Отрезаем мусор начала строки (цифры списков, маркеры, пробелы, пайпы), чтобы увидеть букву
            clean_start = re.sub(r'^[\s*+\-•|#\d.)\];]+', '', line).strip()
            
            # Если строка начинается с заглавной буквы это то что нам нада
            if clean_start and clean_start[0].isupper():
                start = i
                found_good_start = True
                break
                
        # Если заглавную букву не нашли, просто берем стандартный сдвиг
        if not found_good_start:
            start = target_start
            
    return chunks

In [10]:
def process_articles(metadata_path: str, out_dir: str = "./chunks"):
    base = Path(out_dir)
    (base / 'raw').mkdir(parents=True, exist_ok=True)
    (base / 'clean').mkdir(parents=True, exist_ok=True)
    
    articles = json.loads(Path(metadata_path).read_text(encoding='utf-8'))
    meta, g_id = [], 0
    
    for art in articles:
        f_path = Path(art['file'])
        if not f_path.exists(): continue
        
        # Читаем оригинальный файл как есть
        source_text = f_path.read_text(encoding='utf-8')
        
        # Нарезаем строго по строкам
        chunks = chunk_by_lines(source_text, max_lines=30, overlap_lines=4)
        
        for i, r_content in enumerate(chunks):
            # Пропускаем совсем пустые куски
            if len(r_content.strip()) < 15: continue
            
            c_id = f"chunk_{g_id:05d}"
            
            # 1. RAW: Абсолютно нетронутый текст (таблицы и переносы строк сохранены)
            (base / f"raw/{c_id}.md").write_text(r_content, encoding='utf-8')
            
            # 2. CLEAN: Жестко вычищенный текст, пайпы заменены на ';'
            c_content = clean_for_search(r_content)
            (base / f"clean/{c_id}.txt").write_text(c_content, encoding='utf-8')
            
            # 3. Метаданные
            meta.append({
                **art, 
                'chunk_id': c_id, 
                'chunk_index': i,
                'raw_file': str(base / f"raw/{c_id}.md"), 
                'clean_file': str(base / f"clean/{c_id}.txt")
            })
            g_id += 1
            
    (base / "chunks_metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Готово! Создано {g_id} чанков.")

In [11]:
# Запуск
process_articles('articles_metadata.json')

Готово! Создано 1002 чанков.


Далее делаем еще один набор чанков. Следующий набор будет исключительно для работы sparse ретривера, форматируем все чанки в формат, с которым он должен работать.

In [12]:
# Скачиваем стоп-слова для русского языка
nltk.download('stopwords', quiet=True)

True

In [13]:
def prepare_sparse_processor():
    morph = pymorphy3.MorphAnalyzer()
    stop_words = set(stopwords.words('russian'))

    return morph, stop_words

morph, stop_words = prepare_sparse_processor()

In [14]:
def lemmatize_text(text: str, morph, stop_words) -> str:
    # Оставляем только буквы и цифры
    clean_words = re.findall(r'\b[a-zа-яё0-9-]+\b', text.lower())
    
    lemmatized = []
    for word in clean_words:
        if word not in stop_words and not word.isdigit():
            # Приводим слово к нормальной форме
            parsed = morph.parse(word)[0]
            lemmatized.append(parsed.normal_form)
            
    return " ".join(lemmatized)

In [15]:
def process_sparse_directory(out_dir: str = "./chunks"):
    base = Path(out_dir)
    clean_dir = base / 'clean'
    sparse_dir = base / 'clean-sparse'
    sparse_dir.mkdir(parents=True, exist_ok=True)
    
    if not clean_dir.exists():
        print("Ошибка: Папка clean не найдена! Сначала запусти прошлый чанкер.")
        return
        
    morph, stop_words = prepare_sparse_processor()
    
    # Загружаем существующую мету, чтобы дополнить её путями к sparse-файлам
    meta_path = base / "chunks_metadata.json"
    if meta_path.exists():
        metadata = json.loads(meta_path.read_text(encoding='utf-8'))
    else:
        metadata = []
        
    meta_map = {item['chunk_id']: item for item in metadata}
    
    print("Запуск лемматизации для Sparse индекса...")
    count = 0
    
    for file_path in clean_dir.glob("*.txt"):
        chunk_id = file_path.stem
        text = file_path.read_text(encoding='utf-8')
        
        # Лемматизируем
        sparse_content = lemmatize_text(text, morph, stop_words)
        
        # Сохраняем в clean-sparse
        out_file = sparse_dir / f"{chunk_id}.txt"
        out_file.write_text(sparse_content, encoding='utf-8')
        
        # Обновляем метаданные, если этот чанк там есть
        if chunk_id in meta_map:
            meta_map[chunk_id]['sparse_file'] = str(out_file)
            
        count += 1

    # Перезаписываем обновленную мету
    if metadata:
        meta_path.write_text(json.dumps(list(meta_map.values()), ensure_ascii=False, indent=2), encoding='utf-8')
        
    print(f"Успешно обработано и сохранено {count} файлов в директорию 'clean-sparse'.")

In [16]:
# Запуск этапа
process_sparse_directory()

Запуск лемматизации для Sparse индекса...
Успешно обработано и сохранено 1002 файлов в директорию 'clean-sparse'.


Собираем классы ретриверов. Sparse и Dense каждый может найти до 20 вариантов. Гибрид в конце выберит топ 10 основываясь на результатх каждого. Гибрид будет учитывать даже такие варианта, если один из ретриверов будет полностью уверен в нужности чанка, а второй даже не добавит его в свой список, такой чанк имеет право попасть в итоговый список.

In [17]:
class SparseRetriever:
    def __init__(self, chunks_dir: str = "./chunks"):
        self.base_dir = Path(chunks_dir)
        self.sparse_dir = self.base_dir / 'clean-sparse'
        
        # Загружаем метадату
        with open(self.base_dir / "chunks_metadata.json", "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
            
        self.meta_map = {item['chunk_id']: item for item in self.metadata}
        self.chunk_ids = []
        self.corpus = []
        
        self._build_index()
        
    def _build_index(self):
        sparse_files = sorted(list(self.sparse_dir.glob("*.txt")))
        for file_path in sparse_files:
            chunk_id = file_path.stem
            text = file_path.read_text(encoding='utf-8')
            tokens = text.split()
            if tokens:
                self.chunk_ids.append(chunk_id)
                self.corpus.append(tokens)
                
        self.bm25 = BM25Okapi(self.corpus)
        print(f"[Sparse] Индекс собран. Готово к поиску.")

    def search(self, raw_query: str, morph, stop_words, top_k: int = 5, min_score: float = 1.0) -> list:
        from __main__ import lemmatize_text
        
        cleaned_query = lemmatize_text(raw_query, morph, stop_words)
        query_tokens = cleaned_query.split()
        
        if not query_tokens:
            return []
            
        doc_scores = self.bm25.get_scores(query_tokens)
        results = []
        
        top_indices = sorted(range(len(doc_scores)), key=lambda i: doc_scores[i], reverse=True)[:top_k]
        
        for idx in top_indices:
            score = doc_scores[idx]
            if score <= min_score: 
                continue 
            
            c_id = self.chunk_ids[idx]
            chunk_meta = self.meta_map[c_id]
            
            # Вытаскиваем мета информацию из json
            results.append({
                "score": round(float(score), 2),
                "title": chunk_meta.get("title", "Без названия"),
                "url": chunk_meta.get("url", "https://academy.lamoda.ru/"),
                "raw_file": chunk_meta["raw_file"],
                "last_update": chunk_meta.get("last_update", "(Дата неизвестна)")
            })
            
        return results

In [18]:
class DenseRetriever:
    def __init__(self, chunks_dir: str = "./chunks", model_name: str = "cointegrated/rubert-tiny2"):
        self.base_dir = Path(chunks_dir)
        self.clean_dir = self.base_dir / 'clean'
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[Dense] Используем устройство: {self.device.upper()}")
        
        self.model = SentenceTransformer(model_name, device=self.device)
        
        # Загружаем метадату
        with open(self.base_dir / "chunks_metadata.json", "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
        self.meta_map = {item['chunk_id']: item for item in self.metadata}
        
        self.chunk_ids = []
        self.embeddings = None
        
        self._build_index()
        
    def _build_index(self):
        clean_files = sorted(list(self.clean_dir.glob("*.txt")))
        texts_to_embed = []
        
        for file_path in clean_files:
            chunk_id = file_path.stem
            text = file_path.read_text(encoding='utf-8')
            if text.strip():
                self.chunk_ids.append(chunk_id)
                texts_to_embed.append(text)
                
        if not texts_to_embed:
            print("[Dense] Ошибка: Папка clean пуста.")
            return
            
        print(f"[Dense] Кодируем {len(texts_to_embed)} чанков")
        
        # Распараллелим задачу
        self.embeddings = self.model.encode(
            texts_to_embed, 
            convert_to_tensor=True, 
            batch_size=8, 
            show_progress_bar=True
        )
        print("[Dense] Индексация завершена.")

    def search(self, raw_query: str, top_k: int = 5, min_similarity: float = 0.45) -> list:
        if self.embeddings is None:
            return []
            
        query_embedding = self.model.encode(raw_query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.embeddings)[0]
        
        top_results = torch.topk(cos_scores, k=min(top_k, len(cos_scores)))
        
        scores = top_results.values.cpu().tolist()
        indices = top_results.indices.cpu().tolist()
        
        results = []
        for score, idx in zip(scores, indices):
            if score <= min_similarity:
                continue
                
            c_id = self.chunk_ids[idx]
            chunk_meta = self.meta_map[c_id]
            
            results.append({
                "score": round(float(score), 4),
                "title": chunk_meta.get("title", "Без названия"),
                "url": chunk_meta.get("url", "https://academy.lamoda.ru/"),
                "raw_file": chunk_meta["raw_file"],
                "last_update": chunk_meta.get("last_update", "(Дата неизвестна)")
            })
            
        return results

In [19]:
class HybridRetriever:
    def __init__(self, sparse_retriever, dense_retriever):
        self.sparse = sparse_retriever
        self.dense = dense_retriever
        
    def search(self, raw_query: str, morph, stop_words, top_k: int = 10, k_rrf: int = 60, min_rrf_score: float = 0.016) -> list:

        # 1. Запрашиваем ТОП-20 у каждого ретривера
        sparse_res = self.sparse.search(raw_query, morph, stop_words, top_k=20)
        dense_res = self.dense.search(raw_query, top_k=20)
        
        # Если вообще ничего не нашли оба ретривера
        if not sparse_res and not dense_res:
            return []
            
        rrf_scores = {}
        doc_data = {} # Хранилище метаданных
        
        # 2. Считаем баллы для результатов Sparse (BM25)
        for rank, doc in enumerate(sparse_res, start=1):
            doc_id = doc["raw_file"]
            doc_data[doc_id] = doc
            # Формула RRF: 1 / (k + rank)
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))
            
        # 3. Считаем баллы для результатов Dense (rubert-tiny2)
        for rank, doc in enumerate(dense_res, start=1):
            doc_id = doc["raw_file"]
            # Если документа не было в выдаче sparse, сохраняем его метадату
            if doc_id not in doc_data:
                doc_data[doc_id] = doc
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank))
            
        # 4. Сортируем документы по накопленному RRF-скору (от большего к меньшему)
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Фильтруем: оставляем только те чанки, у которых скор равен или выше порога
        filtered_docs = []
        for doc_id, score in sorted_docs:
            if score >= min_rrf_score:
                filtered_docs.append((doc_id, score))
            else:
                # Так как список отсортирован, как только пошел скор ниже порога — дальше идти нет смысла
                break
        
        # 5. Формируем финальный список
        final_results = []
        for doc_id, score in filtered_docs[:top_k]:
            item = doc_data[doc_id].copy()
            item["rrf_score"] = round(score, 4)
            final_results.append(item)
            
        return final_results

In [20]:
# 2. Создаем экземпляры базовых ретриверов
retriever = SparseRetriever()
dense_retriever = DenseRetriever()

# 3. Собираем их в единый гибрид, который ищет функция
hybrid_retriever = HybridRetriever(sparse_retriever=retriever, dense_retriever=dense_retriever)

print("Все поисковые движки успешно инициализированы и готовы к работе!")

[Sparse] Индекс собран. Готово к поиску.
[Dense] Используем устройство: CUDA


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Dense] Кодируем 1002 чанков


Batches:   0%|          | 0/126 [00:00<?, ?it/s]

[Dense] Индексация завершена.
Все поисковые движки успешно инициализированы и готовы к работе!


In [21]:
# Инициализируем новый клиент по официальному гайду Google
client = genai.Client(api_key=GOOGLE_API_KEY.strip())

In [22]:
def run_lamoda_rag(user_question: str, model_name: str = "gemini-flash-lite-latest"):
    # Параметры генерации чтобы не шизил
    config = types.GenerateContentConfig(
        temperature=0.4,
        top_p=0.95,
        top_k=40
    )
    
    # 2. Генерируем 3 поисковых запроса через LLM
    query_prompt = (
        f"Пользователь хочет найти в базе знаний ответ на вопрос: '{user_question}'. "
        f"Сгенерируй ровно 3 разных, точных поисковых запроса на русском языке, которые помогут "
        f"найти нужные документы. Пиши строго по одному запросу на строке, "
        f"без нумерации, маркеров списков, кавычек и лишних пояснений."
    )
    
    try:
        # Вызываем модель через client.models.generate_content
        query_response = client.models.generate_content(
            model=model_name,
            contents=query_prompt,
            config=config
        ).text.strip()
        generated_queries = [q.strip() for q in query_response.split('\n') if q.strip()][:3]
    except Exception as e:
        generated_queries = [user_question]

    # Выводим что происходит
    print(f"Вопрос пользователя: {user_question}")
    print("Сформированные запросы к базе знаний:")
    
    # Собираем чанки по каждому запросу отдельно и сразу логируем результат
    all_chunks = []
    for q in generated_queries:
        print(f"- {q}")
        res = hybrid_retriever.search(q, morph, stop_words, top_k=10)
        print(f"  - Получено {len(res)} ответов")
        all_chunks.extend(res)
        
    # Дедупликация по уникальному файлу raw_file
    seen_files = set()
    unique_chunks = []
    for chunk in all_chunks:
        file_id = chunk["raw_file"]
        if file_id not in seen_files:
            seen_files.add(file_id)
            unique_chunks.append(chunk)
            
    # Если ни один чанк не подошел
    if not unique_chunks:
        print("\nК сожалению, у меня нет информации по этому вопросу, попробуйте переформулировать ваш вопрос.")
        return

    # Подготовка контекста и сбор метаданных для вывода
    context_parts = []
    articles_found = []
    used_sources = []
    
    for doc in unique_chunks:
        raw_path = Path(doc["raw_file"])
        full_text = raw_path.read_text(encoding='utf-8')
        
        context_parts.append(f"Статья: {doc['title']}\nКонтекст: {full_text}\n---")
        
        if doc['title'] not in articles_found:
            articles_found.append(doc['title'])
            
        last_update = doc.get("last_update") or "(Дата неизвестна)"
        source_info = {
            "title": doc["title"],
            "url": doc["url"],
            "date": last_update
        }
        if source_info not in used_sources:
            used_sources.append(source_info)
            
    context_str = "\n".join(context_parts)

    # Продолжаем вывод логов: пишем агрегированную инфу по уникальным чанкам
    print(f"Получил {len(unique_chunks)} уникальных ответов из:")
    for title in articles_found:
        print(f"  - {title}")

    # Сам ответ модели
    final_prompt = (
        f"Ты — ассистент поддержки Академии селлеров Lamoda. Ответь на вопрос пользователя, "
        f"используя только предоставленный контекст. Если в контексте нет прямого ответа на вопрос, "
        f"напиши фразу: 'К сожалению, у меня нет информации по этому вопросу, попробуйте переформулировать ваш вопрос.'\n\n"
        f"Контекст из базы знаний:\n{context_str}\n\n"
        f"Вопрос пользователя: {user_question}\n"
        f"Ответ:"
    )
    
    # Вызываем финальный ответ через новый синтаксис
    final_answer = client.models.generate_content(
        model=model_name,
        contents=final_prompt,
        config=config
    ).text.strip()
    
    # Завершающая часть вывода
    print("\nОтвет готов!")
    print(final_answer)
    
    print("\nДля более точной информации, воспользуйтесь источниками:")
    for src in used_sources:
        print(f"  - [{src['title']}]({src['url']}) от {src['date']}")

In [25]:
run_lamoda_rag("Что дает Lamoda API, что можно через нее сделать?") # Сюда вводите ваш запрос

Вопрос пользователя: Что дает Lamoda API, что можно через нее сделать?
Сформированные запросы к базе знаний:
- возможности Lamoda API
  - Получено 9 ответов
- функционал Lamoda API для партнеров
  - Получено 10 ответов
- что можно делать через API Lamoda
  - Получено 8 ответов
Получил 19 уникальных ответов из:
  - Точки входа (endpoints) API
  - Быстрый старт API
  - О Lamoda API
  - Сравнение Lamoda Seller Partner API и Lamoda B2B Platform API
  - Остатки FBO в API
  - FBS vs FBO: сравнение моделей в API
  - Обработка ошибок в API
  - Создание и редактирование товаров через API
  - Статусы FBO в API
  - Lamoda B2B Platform API
  - История изменений API
  - Lamoda Seller partner API
  - Lamoda Seller partner API REST
  - Версии API
  - Поддержка в API
  - Настройка вебхуков в API
  - Бронирование слота на поставку

Ответ готов!
Lamoda API — это набор интерфейсов для продавцов-партнёров, который позволяет автоматизировать управление следующими процессами:

*   **Товары:** создание и ред